# DarkGuard SFT on Kaggle (Unsloth + TRL)

This notebook runs Stage 1 SFT for DarkGuard on Kaggle GPU using the existing `sft_training/train_sft.py` script.

## Before running
- Enable GPU in Kaggle Notebook settings.
- Ensure your repo is available in `/kaggle/working/RL_Env`.
- Add Kaggle Secrets:
  - `HF_TOKEN`
  - `WANDB_API_KEY` (optional)

## Complete Kaggle Flow (required extras)

Use the cells below to make the notebook fully production-ready:

1. Write `sft_training/.env` from Kaggle Secrets (for `load_dotenv`).
2. Train one or both roles with resume support.
3. Optionally upload adapter/merged artifacts to Hugging Face Hub.
4. Zip outputs for Kaggle download.

In [ ]:
# Full setup (runtime check, repo path, deps, secrets, dataset validation, .env write)
import os
import subprocess
from pathlib import Path


def run(cmd: str):
    print(f"\n[RUN] {cmd}")
    subprocess.run(cmd, shell=True, check=True)


print("Python:", subprocess.check_output("python3 --version", shell=True).decode().strip())
run("nvidia-smi || true")

# Adjust if your repo folder differs in Kaggle.
REPO_ROOT = Path('/kaggle/working/RL_Env')
assert REPO_ROOT.exists(), f"Repo not found at {REPO_ROOT}. Upload/clone repo first."
os.chdir(REPO_ROOT)
print('Working directory:', Path.cwd())

run('python3 -m pip install --upgrade pip')
run('python3 -m pip install -r sft_training/requirements.txt')

# Load Kaggle secrets.
try:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN').strip()
    print('HF_TOKEN loaded from Kaggle Secrets.')

    try:
        wkey = secrets.get_secret('WANDB_API_KEY')
        if wkey:
            os.environ['WANDB_API_KEY'] = wkey.strip()
            print('WANDB_API_KEY loaded from Kaggle Secrets.')
    except Exception:
        print('WANDB_API_KEY not set (optional).')
except Exception as e:
    raise RuntimeError('Add HF_TOKEN in Kaggle Add-ons -> Secrets.') from e

# Validate required dataset files.
required = [
    REPO_ROOT / 'darkguard_preprocessed/consumer_train.jsonl',
    REPO_ROOT / 'darkguard_preprocessed/consumer_val.jsonl',
    REPO_ROOT / 'darkguard_preprocessed/designer_train.jsonl',
    REPO_ROOT / 'darkguard_preprocessed/designer_val.jsonl',
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(f'Missing dataset files: {missing}')
print('All required JSONL files found.')

# Write .env for train_sft.py (which uses load_dotenv()).
sft_env = REPO_ROOT / 'sft_training/.env'
lines = [
    f"HF_TOKEN={os.environ.get('HF_TOKEN', '').strip()}",
    f"WANDB_PROJECT={os.environ.get('WANDB_PROJECT', 'darkguard-sft').strip()}",
]
if os.environ.get('WANDB_API_KEY'):
    lines.append(f"WANDB_API_KEY={os.environ['WANDB_API_KEY'].strip()}")

sft_env.write_text("\n".join(lines) + "\n", encoding='utf-8')
print('Wrote', sft_env)
print(sft_env.read_text())

In [ ]:
# Complete training runner with role selection + resume support.

ROLES_TO_TRAIN = ['consumer', 'designer']  # choose ['consumer'] / ['designer'] / both
RESUME_CHECKPOINTS = {
    # Example: 'consumer': str(REPO_ROOT / 'outputs/consumer/checkpoint-100')
}

MODEL_NAME = 'unsloth/Qwen3-4B-Thinking-2507-FP8'
MAX_SEQ_LENGTH = 3072
BATCH_SIZE = 4
GRAD_ACCUM = 8
LEARNING_RATE = 2e-4
EPOCHS = 2
LORA_R = 32
LOAD_IN_4BIT = 'true'
USE_WANDB = False

for role in ROLES_TO_TRAIN:
    train_file = REPO_ROOT / f'darkguard_preprocessed/{role}_train.jsonl'
    val_file = REPO_ROOT / f'darkguard_preprocessed/{role}_val.jsonl'
    cmd = (
        'python3 sft_training/train_sft.py '
        f'--role {role} '
        f'--model_name {MODEL_NAME} '
        f'--train_file {train_file} '
        f'--val_file {val_file} '
        f'--output_dir {REPO_ROOT / "outputs"} '
        f'--max_seq_length {MAX_SEQ_LENGTH} '
        f'--batch_size {BATCH_SIZE} '
        f'--grad_accum {GRAD_ACCUM} '
        f'--learning_rate {LEARNING_RATE} '
        f'--epochs {EPOCHS} '
        f'--lora_r {LORA_R} '
        f'--load_in_4bit {LOAD_IN_4BIT} '
    )

    if USE_WANDB:
        cmd += '--use_wandb '

    ckpt = RESUME_CHECKPOINTS.get(role)
    if ckpt:
        cmd += f'--resume_from_checkpoint {ckpt} '

    print(f'\\n=== Training role: {role} ===')
    run(cmd)

print('Training run(s) complete.')

In [ ]:
# Package outputs so you can download from Kaggle output panel.
archive_path = Path('/kaggle/working/darkguard_sft_outputs.zip')
run(f'cd {REPO_ROOT} && zip -r {archive_path} outputs')
print('Created archive:', archive_path)

In [ ]:
# Optional: upload artifacts to Hugging Face Hub model repos.
# Set ENABLE_UPLOAD=True and replace repo IDs before running.

ENABLE_UPLOAD = False
CONSUMER_ADAPTER_REPO = 'Jyo-K/darkguard-consumer-adapter'
DESIGNER_ADAPTER_REPO = 'Jyo-K/darkguard-designer-adapter'

if ENABLE_UPLOAD:
    from huggingface_hub import HfApi

    api = HfApi(token=os.environ.get('HF_TOKEN'))

    consumer_adapter = REPO_ROOT / 'outputs/consumer/adapter'
    designer_adapter = REPO_ROOT / 'outputs/designer/adapter'

    if consumer_adapter.exists():
        api.create_repo(CONSUMER_ADAPTER_REPO, repo_type='model', exist_ok=True)
        api.upload_folder(folder_path=str(consumer_adapter), repo_id=CONSUMER_ADAPTER_REPO, repo_type='model')
        print('Uploaded consumer adapter ->', CONSUMER_ADAPTER_REPO)

    if designer_adapter.exists():
        api.create_repo(DESIGNER_ADAPTER_REPO, repo_type='model', exist_ok=True)
        api.upload_folder(folder_path=str(designer_adapter), repo_id=DESIGNER_ADAPTER_REPO, repo_type='model')
        print('Uploaded designer adapter ->', DESIGNER_ADAPTER_REPO)
else:
    print('Skipping upload. Set ENABLE_UPLOAD=True to enable.')